# Sesión 36 · La librería Datasets

Explora, carga, transforma y publica datasets del Hub.

> Formato: ~30 min explicación · ~75 min notebook · ~15 min reto.

In [ ]:
%pip install -q transformers datasets evaluate accelerate matplotlib

---
## 1 · Explorar el Hub de datasets

El Hub tiene más de 100 000 datasets públicos. Podemos listarlos e inspeccionarlos
con la API de `huggingface_hub`.

In [ ]:
from huggingface_hub import list_datasets

# El generador es lazy: solo descarga metadatos cuando se itera
gen = list_datasets()
primeros = [next(gen).id for _ in range(5)]
print("Primeros 5 datasets:", primeros)

In [ ]:
from datasets import load_dataset

# Cargar IMDB completo (descarga ~80 MB la primera vez)
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)
print("\nPrimer ejemplo:")
print(dataset["train"][0])

### 1a · Streaming: iterar sin descargar todo

`streaming=True` evita descargar el dataset completo. Los datos llegan por lotes
a medida que se iteran. Imprescindible en Colab con datasets de varios GB.

In [ ]:
dataset_stream = load_dataset("stanfordnlp/imdb", streaming=True)
print(type(dataset_stream["train"]))  # IterableDataset, no Dataset

print("\nPrimeros 3 ejemplos en streaming:")
for i, ejemplo in enumerate(dataset_stream["train"]):
    if i >= 3:
        break
    print(f"  label={ejemplo['label']}  texto={ejemplo['text'][:60]}...")

### 1b · Visualizar la distribución de etiquetas

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Dataset de noticias financieras en Twitter (20 categorías)
ds_fin = load_dataset("zeroshot/twitter-financial-news-topic")
print(ds_fin)

topicos = {
    0: "Analyst Update", 1: "Fed/Central Banks", 2: "Company News",
    3: "Debt", 4: "Dividend", 5: "Earnings", 6: "Energy",
    7: "Financials", 8: "Currencies", 9: "General News",
    10: "Gold/Metals", 11: "IPO", 12: "Legal", 13: "M&A",
    14: "Macro", 15: "Markets", 16: "Politics",
    17: "Personnel", 18: "Stock Commentary", 19: "Stock Movement",
}

conteo = Counter(ds_fin["train"]["label"])
etiquetas_ord = [topicos[k] for k in sorted(conteo)]
valores_ord   = [conteo[k] for k in sorted(conteo)]

plt.figure(figsize=(12, 5))
plt.bar(etiquetas_ord, valores_ord, color="steelblue", alpha=0.8)
plt.xticks(rotation=90)
plt.title("Distribución de categorías — Twitter Financial News")
plt.ylabel("Número de tweets")
plt.tight_layout()
plt.show()

---
## 2 · Subconjunto y tokenización con `.map()`

Para el fine-tuning usaremos solo 1 000 ejemplos de entrenamiento (más rápido
en Colab). En un proyecto real usarías todo el split.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train = dataset["train"].shuffle(seed=42).select(range(1000))
test  = dataset["test"].shuffle(seed=42).select(range(200))

def tokenizar(ejemplos):
    return tokenizer(
        ejemplos["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train = train.map(tokenizar, batched=True)
test  = test.map(tokenizar, batched=True)

print(f"Train: {len(train)} ejemplos | Test: {len(test)} ejemplos")
print("Columnas:", train.column_names)

---
## 3 · Crear, persistir y publicar datasets propios

In [ ]:
from datasets import Dataset

datos = [
    {"texto": "Me encanta este producto, funciona perfectamente.",  "etiqueta": 1},
    {"texto": "Pésima calidad, se rompió al segundo día.",          "etiqueta": 0},
    {"texto": "Llego antes de lo previsto, muy contento.",          "etiqueta": 1},
    {"texto": "El embalaje estaba dañado y faltaban piezas.",       "etiqueta": 0},
]

ds_propio = Dataset.from_list(datos)
print(ds_propio)
print(ds_propio[0])

# Guardar en formato Apache Arrow (eficiente para datasets grandes)
ds_propio.save_to_disk("mi_dataset_arrow")

# Exportar a Parquet (interoperable con pandas, Spark…)
ds_propio.to_parquet("mi_dataset.parquet")

# También: to_csv(), to_json()
print("\nDataset guardado en Arrow y Parquet.")

# Publicar en el Hub (requiere token de escritura)
# ds_propio.push_to_hub("tu-usuario/mi-dataset-reseñas")

---
## Práctica

Prepara un dataset de **reseñas en español** tokenizado y particionado (se usará en la sesión 38).

## Reto

Sube al Hub un dataset propio (p. ej. `FuelConsumptionCo2` del curso de ML) con su *dataset card*.